# Session 4 · Visualisation and data storytelling
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/urosgodnov/neoma-python-2627/blob/main/notebooks/S04_lecture_visualisation.ipynb)  ·  *Python Programming for Business Analytics · NEOMA 2026/27 · dr. Uroš Godnov*

**Live on Zoom · 3 hours.** From default charts to charts that make a decision easy. We use **seaborn** for the statistics and **matplotlib** for the finishing touches. Data: Gapminder (as in the R course) and the penguins.

| Part | Time | Topic |
|---|---|---|
| A1 | 0:00 | How Python draws: figure, axes, seaborn |
| A2 | 0:15 | Your brand theme |
| A3 | 0:35 | The four essential charts |
| ☕ | **1:30** | **Break · 10 minutes** |
| B1 | 1:40 | Facets: small multiples |
| B2 | 2:00 | Storytelling makeover |
| B3 | 2:25 | A dashboard in one figure |
| B4 | 2:45 | 🤖 AI charts, interactive bonus, wrap-up |

In [ ]:
# ⚙️ Setup: run this cell first (works in Google Colab and in Positron)
from pathlib import Path
import numpy as np
import pandas as pd

REPO = "https://raw.githubusercontent.com/urosgodnov/neoma-python-2627/main/"


def data_path(name):
    """Use the local course folder if you have it, otherwise read the file from GitHub."""
    for base in (Path("data"), Path("../data")):
        if (base / name).exists():
            return base / name
    return REPO + "data/" + name


pd.set_option("display.max_columns", 30)
print("Python ready · pandas", pd.__version__)
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

gapminder = pd.read_csv(data_path("gapminder.csv"))
penguins = pd.read_csv(data_path("penguins.csv"))
gap07 = gapminder.query("year == 2007")
gapminder.head()

---
## A1 · How Python draws

- **matplotlib**: the engine. A `Figure` (the canvas) holds one or more `Axes` (the plots). You control every pixel.
- **seaborn**: the statistician on top. Give it a DataFrame and column names, like `aes()` in ggplot2.

| ggplot2 | seaborn / matplotlib |
|---|---|
| `ggplot(df, aes(x, y, colour = g))` | `sns.scatterplot(data=df, x=..., y=..., hue=g)` |
| `geom_point()`, `geom_line()`, `geom_col()` | `scatterplot`, `lineplot`, `barplot` |
| `geom_histogram()`, `geom_boxplot()`, `geom_violin()` | `histplot`, `boxplot`, `violinplot` |
| `geom_smooth(method = "lm")` | `sns.regplot()` / `sns.lmplot()` |
| `facet_wrap(~ g)` | `sns.relplot(..., col=g, col_wrap=3)` |
| `labs(title = ...)` | `ax.set(title=..., xlabel=..., ylabel=...)` |
| `scale_x_log10(labels = dollar)` | `ax.set_xscale("log")` + `ax.xaxis.set_major_formatter(...)` |
| `theme_set()` | `sns.set_theme()` / `plt.rcParams` |
| `patchwork` | `plt.subplots()` / `plt.subplot_mosaic()` |

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.scatterplot(data=gap07, x="gdpPercap", y="lifeExp", ax=ax)
plt.show()

💬 **Discuss:** what is wrong with this chart for a manager? List three things.

---
## A2 · Your brand theme

Set it **once**, at the top of every notebook. One palette, one font size, no clutter. (🆚 ggplot2 4.0 `theme(ink, paper, accent)` + `theme_set()`.)

In [ ]:
BRAND = ["#1F4E79", "#E07A1F", "#3A9D6E", "#B5446E", "#7A7A7A"]   # five colours, colour-blind friendly order
ACCENT, GREY = "#C0392B", "#BBBBBB"

sns.set_theme(
    style="whitegrid",
    palette=BRAND,
    rc={
        "figure.figsize": (8, 4.5),
        "figure.dpi": 110,
        "axes.titlesize": 14,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "grid.color": "#EEEEEE",
    },
)
dollars = mticker.FuncFormatter(lambda x, _: f"${x:,.0f}")

---
## A3 · The four essential charts

### 1 · Scatter: relationship
Log scale for money, dollar labels, colour = continent, size = population, one trend line for all countries.

In [ ]:
fig, ax = plt.subplots()
sns.scatterplot(data=gap07, x="gdpPercap", y="lifeExp", hue="continent", size="pop",
                sizes=(15, 600), alpha=0.7, ax=ax)
sns.regplot(data=gap07, x="gdpPercap", y="lifeExp", scatter=False, logx=True,
            color="black", line_kws={"lw": 1, "ls": "--"}, ax=ax)
ax.set_xscale("log")
ax.xaxis.set_major_formatter(dollars)
ax.set(title="Richer countries live longer, with diminishing returns per extra dollar",
       xlabel="GDP per capita (log scale)", ylabel="Life expectancy (years)")
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[1:6], labels[1:6], title=None, frameon=False, loc="lower right")
plt.show()

🔮 **Predict:** what changes if you remove `logx=True`? Try it.

### 2 · Bars: comparison
Horizontal bars, sorted, labels inside the bars: no need to read an axis.

In [ ]:
top10 = gap07.nlargest(10, "pop").assign(pop_m=lambda d: d["pop"] / 1e6)

fig, ax = plt.subplots()
sns.barplot(data=top10, y="country", x="pop_m", color=BRAND[0], ax=ax)
ax.bar_label(ax.containers[0], fmt="%.0f M", padding=4, fontweight="bold")
ax.set(title="China and India together have more people than the next eight countries",
       xlabel=None, ylabel=None, xticks=[])
ax.grid(False)
plt.show()

### 3 · Lines: change over time
Direct labels at the end of each line instead of a legend: the eye does not travel.

In [ ]:
five = ["Slovenia", "France", "China", "Botswana", "India"]
d5 = gapminder[gapminder["country"].isin(five)]

fig, ax = plt.subplots()
sns.lineplot(data=d5, x="year", y="lifeExp", hue="country", marker="o", legend=False, ax=ax)
for i, (country, g) in enumerate(d5.groupby("country")):
    last = g.loc[g["year"].idxmax()]
    ax.annotate(country, (last["year"], last["lifeExp"]), xytext=(6, 0),
                textcoords="offset points", va="center", color=ax.lines[i].get_color())
ax.set(title="Botswana lost a decade of life expectancy in the 1990s",
       xlabel=None, ylabel="Life expectancy (years)", xlim=(1950, 2015))
plt.show()

### 4 · Distributions: spread
Violin + quartiles + every observation.

In [ ]:
fig, ax = plt.subplots()
sns.violinplot(data=penguins, x="species", y="body_mass", hue="species", inner="quart",
               cut=0, alpha=0.35, legend=False, ax=ax)
sns.stripplot(data=penguins, x="species", y="body_mass", color="black", size=2.5, alpha=0.5, ax=ax)
ax.set(title="Gentoo penguins are about 1.4 kg heavier than the other two species",
       xlabel=None, ylabel="Body mass (g)")
plt.show()

🧪 **Try it (5 min):** a histogram of life expectancy in 2007 with `sns.histplot(..., hue="continent", multiple="stack")`. Write a title that states an insight.

In [ ]:
# your code here

---
## ☕ Break · 10 minutes
---

## B1 · Facets: small multiples
A **label dictionary** keeps axis titles consistent across all charts (🆚 `labs(dictionary = ...)`).

In [ ]:
LABELS = {"lifeExp": "Life expectancy (years)", "gdpPercap": "GDP per capita",
          "pop": "Population", "continent": "Continent", "year": "Year"}

g = sns.relplot(data=gapminder, x="year", y="lifeExp", col="continent", col_wrap=3,
                kind="line", units="country", estimator=None, color=GREY, lw=0.7,
                height=2.6, aspect=1.3)
for continent, ax in g.axes_dict.items():
    med = gapminder[gapminder["continent"] == continent].groupby("year")["lifeExp"].median()
    ax.plot(med.index, med.values, color=ACCENT, lw=2.5)
g.set_titles("{col_name}")
g.set_axis_labels(LABELS["year"], LABELS["lifeExp"])
g.figure.suptitle("Every continent gained life expectancy; Africa's progress stalled after 1990 (red = median)",
                  x=0.02, ha="left", fontweight="bold", y=1.04)
plt.show()

---
## B2 · Storytelling makeover
**Question:** which countries had a **lower** life expectancy in 2007 than in 1987?

Recipe: grey for context, one accent colour for the story, labels instead of a legend, a title that **states the insight**.

In [ ]:
change = (
    gapminder.query("year in [1987, 2007]")
    .pivot(index=["country", "continent"], columns="year", values="lifeExp")
    .assign(change=lambda d: d[2007] - d[1987])
    .reset_index()
    .sort_values("change")
)
losers = change[change["change"] < 0]
print(len(losers), "countries:", ", ".join(losers["country"]))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(change["change"], change[2007], color=GREY, s=25)
ax.scatter(losers["change"], losers[2007], color=ACCENT, s=45)
for _, r in losers.iterrows():
    ax.annotate(r["country"], (r["change"], r[2007]), xytext=(4, 2), textcoords="offset points",
                fontsize=8, color=ACCENT)
ax.axvline(0, color="black", lw=0.8)
ax.set(title=f"{len(losers)} countries, most of them in sub-Saharan Africa, lived shorter lives in 2007 than in 1987",
       xlabel="Change in life expectancy 1987 → 2007 (years)", ylabel="Life expectancy 2007 (years)")
ax.title.set_fontsize(12)
plt.show()

💬 **Discuss:** what is the likely cause? Which chart from A3 would you show next to support the story?

---
## B3 · A dashboard in one figure
`plt.subplot_mosaic()` lays out panels by name (🆚 `patchwork`: `(p1 | p2) / p3`).

In [ ]:
fig, axd = plt.subplot_mosaic([["A", "B"], ["C", "C"]], figsize=(11, 7.5), layout="constrained")

sns.scatterplot(data=gap07, x="gdpPercap", y="lifeExp", hue="continent", alpha=0.7, ax=axd["A"], legend=False)
axd["A"].set_xscale("log")
axd["A"].xaxis.set_major_formatter(dollars)
axd["A"].set(title="Wealth and health", xlabel="GDP per capita", ylabel="Life expectancy")

cont = gap07.groupby("continent")["lifeExp"].median().sort_values()
axd["B"].barh(cont.index, cont.values, color=BRAND[0])
axd["B"].bar_label(axd["B"].containers[0], fmt="%.0f", padding=3)
axd["B"].set(title="Median life expectancy 2007", xlabel="Years")

trend = gapminder.groupby(["year", "continent"], as_index=False)["lifeExp"].median()
sns.lineplot(data=trend, x="year", y="lifeExp", hue="continent", marker="o", ax=axd["C"])
axd["C"].set(title="Fifty years of progress", xlabel=None, ylabel="Median life expectancy")
axd["C"].legend(frameon=False, ncol=5, loc="lower right")

for tag, ax in axd.items():
    ax.text(-0.02, 1.08, tag, transform=ax.transAxes, fontsize=14, fontweight="bold", va="bottom", ha="right")
fig.suptitle("The world got healthier, but Africa still lags about 25 years behind Europe", fontsize=15,
             fontweight="bold", x=0.01, ha="left")
fig.text(0.99, -0.02, "Source: Gapminder (gapminder.org), via the gapminder R package", ha="right", fontsize=8, color="grey")
fig.savefig("dashboard.png", dpi=200, bbox_inches="tight")
plt.show()

---
## B4 · 🤖 AI charts and an interactive bonus

1. **Colab:** click Gemini ✦ and ask: *"Using the DataFrame gap07, create a bubble chart of GDP per capita vs life expectancy by continent."*
2. **Critique it with today's checklist:** Does the title state an insight? Is money on a log scale with units? Is there clutter? Is colour used for meaning?
3. Improve the AI's chart in two steps. That is the job AI cannot do for you: **deciding what the chart should say**.

**Interactive bonus** with plotly express (pre-installed in Colab):

In [ ]:
import plotly.express as px

px.scatter(gapminder, x="gdpPercap", y="lifeExp", size="pop", color="continent", hover_name="country",
           animation_frame="year", log_x=True, size_max=45, range_y=[25, 90],
           labels=LABELS, title="Fifty years of wealth and health")

---
**Checklist for every chart** (you will be graded on it in the final project)
1. The title states the **insight**, not the variables.
2. Axis labels with **units**; money on a log scale when it spans orders of magnitude.
3. Colour has a **meaning** (category or highlight), never decoration.
4. Direct labels instead of a legend when possible.
5. A source line.

**Before Session 5:** practice notebook **S04_practice**, the **Session 4 quiz**, and block 3 hours for the asynchronous Session 5.